# Preprocessing — M-Sand SDL

Takes the raw dataset (`01_RAW_dataset.csv`, 344 rows x 98 template columns) and produces
the two model-ready files:

- `04_workability_dataset.csv` — one row per mix, cone slump only
- `03_strength_dataset.csv` — **one row per mix**, 28-day target, **the same mixes as workability**

**Rules that must not change:**
1. Never impute. A blank means "not measured", not zero.
2. Flow is not cone slump. Never put them in the same column.
3. A statistical outlier is not automatically an error. Remove only what is physically impossible.
4. Keep every original column. Add derived features; delete nothing.
5. **One physical mix = one row.** Test ages are columns, never extra rows.

**Project choice (not an IS rule):** strength and workability use the same set of mixes, so
the two models describe the same concrete. Mixes with cube results but no cone slump are left
out of both files and listed below.

In [1]:
import pandas as pd, numpy as np
pd.set_option('display.width', 200)

raw = pd.read_csv('data/01_RAW_dataset.csv')
print(f"raw: {len(raw)} rows x {len(raw.columns)} columns")
print(raw.source_file.value_counts())


raw: 344 rows x 98 columns
source_file
Data_Meet_RMC               256
Combined_Msand_Dataset_1     70
plant_a_pdfs          18
Name: count, dtype: int64


## 1. Numeric coercion

Template cells arrive as text in places. Coerce the numeric columns, leaving
anything unparseable as NaN rather than guessing a value.


In [2]:
N = lambda c: pd.to_numeric(raw[c], errors='coerce')

d = raw.copy()
d['cement_kg_m3']  = N('cement')
d['flyash_kg_m3']  = N('flyash').fillna(0)      # 0 = confirmed none, these sources always state it
d['ggbs_kg_m3']    = N('ggbs').fillna(0)
d['other_scm']     = N('other_scm').fillna(0)
d['water_kg_m3']   = N('water_batched')
d['sp_kg_m3']      = N('admixture1')

# coarse aggregate is split across size columns in the template
d['coarse_agg_kg_m3'] = (N('agg_40mm').fillna(0) + N('agg_20mm').fillna(0)
                         + N('agg_10mm').fillna(0) + N('agg_other').fillna(0))
d['fine_agg_kg_m3']   = N('crushed_sand').fillna(0) + N('natural_sand').fillna(0)

print(d[['cement_kg_m3','water_kg_m3','coarse_agg_kg_m3','fine_agg_kg_m3']].describe().round(1))


       cement_kg_m3  water_kg_m3  coarse_agg_kg_m3  fine_agg_kg_m3
count         344.0        344.0             344.0           344.0
mean          345.7        162.3            1068.1           820.1
std            52.4          8.3              91.4            63.9
min            90.0        125.0             720.0           397.0
25%           329.4        158.7            1036.9           798.6
50%           344.2        165.0            1082.8           817.8
75%           368.9        167.5            1133.3           839.5
max           500.0        182.0            1212.0          1042.0


## 2. Derived features

Ratios carry more physical meaning than raw masses. Water-binder ratio is the single
strongest strength driver (Abrams' law); paste fraction and aggregate ratios drive workability.


In [3]:
d['binder_kg_m3']     = d.cement_kg_m3 + d.flyash_kg_m3 + d.ggbs_kg_m3 + d.other_scm
d['wb_ratio']         = (d.water_kg_m3 / d.binder_kg_m3).round(4)
d['scm_frac']         = (d.flyash_kg_m3 + d.ggbs_kg_m3 + d.other_scm) / d.binder_kg_m3
d['sp_pct_binder']    = d.sp_kg_m3 / d.binder_kg_m3 * 100
d['agg_binder_ratio'] = (d.coarse_agg_kg_m3 + d.fine_agg_kg_m3) / d.binder_kg_m3
d['fine_frac_of_agg'] = d.fine_agg_kg_m3 / (d.coarse_agg_kg_m3 + d.fine_agg_kg_m3)
d['msand_frac']       = N('crushed_sand').fillna(0) / d.fine_agg_kg_m3
d['paste_frac']       = ((d.binder_kg_m3 + d.water_kg_m3) /
                         (d.binder_kg_m3 + d.water_kg_m3 + d.coarse_agg_kg_m3 + d.fine_agg_kg_m3))
d['sand_absorption']  = N('sand_absorption')
d['sand_fines_75']    = N('sand_fines_75')

print(f"w/c range: {d.wb_ratio.min():.3f} - {d.wb_ratio.max():.3f}")


w/c range: 0.208 - 0.654


## 3. Design groups — needed for honest validation

Rows that share the same mix quantities are **batches of one design**, not independent
experiments. Data_Meet has 5 designs batched ~50 times each.

If these are split across train and test, the model is tested on a recipe it has already
seen and the score is inflated. Grouping is what makes leave-one-design-out possible.


In [4]:
mix_key = ['cement','flyash','ggbs','water_batched','agg_20mm','agg_10mm','crushed_sand']
d['design_grp'] = np.where(
    d.source_file == 'Data_Meet_RMC',
    d.design_id.astype(str),                              # plant already assigns a design id
    d[mix_key].astype(str).agg('|'.join, axis=1))         # otherwise derive from quantities

print(f"{len(d)} rows -> {d.design_grp.nunique()} distinct designs")
print(d.groupby('source_file').design_grp.nunique())


344 rows -> 79 distinct designs
source_file
Combined_Msand_Dataset_1    58
Data_Meet_RMC                5
plant_a_pdfs         16
Name: design_grp, dtype: int64


## 4. EDA — remove only the physically impossible

Three categories, handled differently:

| category | action | why |
|---|---|---|
| physically impossible | **remove** | water below 110 kg/m3 is not castable concrete |
| mortar (no coarse aggregate) | **exclude from model** | different material, kept in the file flagged |
| statistical outlier | **keep** | a 75 MPa mix is real high-grade concrete, not an error |

Blind IQR deletion would destroy the high-grade tail, which is the data we are shortest of.


In [5]:
flags = pd.DataFrame(index=d.index)
flags['impossible'] = ((d.cement_kg_m3 <= 0) | (d.water_kg_m3 < 110) |
                       (d.wb_ratio <= 0.15) | (d.wb_ratio > 1.2))
flags['mortar'] = d.coarse_agg_kg_m3 == 0

total = (d.binder_kg_m3 + d.water_kg_m3 + d.coarse_agg_kg_m3
         + d.fine_agg_kg_m3 + d.sp_kg_m3.fillna(0))
flags['mass_suspect'] = (total < 2000) | (total > 2800)      # flagged, not removed

remove = flags.impossible | flags.mortar
print(f"impossible  : {flags.impossible.sum()}")
print(f"mortar      : {flags.mortar.sum()}")
print(f"mass flagged: {flags.mass_suspect.sum()}  (kept)")
print(f"removed     : {remove.sum()} -> {(~remove).sum()} rows remain")

d['eda_flag'] = np.where(flags.impossible, 'REMOVE-impossible',
                 np.where(flags.mortar, 'REMOVE-mortar',
                  np.where(flags.mass_suspect, 'FLAG-mass', 'ok')))
d = d[~remove].reset_index(drop=True)


impossible  : 0
mortar      : 0
mass flagged: 1  (kept)
removed     : 0 -> 344 rows remain


## 5. Workability dataset — cone slump only

Built **first**, because it defines which mixes both files use.

**Flow-table values must never enter the slump column.** Ten of the Source A mixes are
self-compacting and measured by flow spread (490-630 mm). A slump cone is 300 mm tall,
so anything above ~250 mm is not a cone measurement.

> **Open issue — not changed here:** IS 1199 (Part 2):2018 cl 4.3.3 gives the valid slump
> range as 10-210 mm. This notebook keeps up to 250 mm, so a few rows may sit outside the
> valid range. The count is printed below. Decide separately whether to tighten the cut-off;
> doing so would change the 299.

In [6]:
d['slump_mm'] = N('slump')
work = d[d.slump_mm.notna() & (d.slump_mm <= 250)].reset_index(drop=True)

print(f"workability: {len(work)} mixes, {work.design_grp.nunique()} designs")
print(f"slump range: {work.slump_mm.min():.0f} - {work.slump_mm.max():.0f} mm")
print(work.source_file.value_counts())
print(f"\nrows above the IS 1199 (Part 2) 210 mm limit: {(work.slump_mm > 210).sum()}  (kept - see open issue)")

s45 = N('slump_45min')
print(f"rows with 45-min slump: {s45.notna().sum()} (one plant, 5 designs - descriptive only)")

workability: 299 mixes, 41 designs
slump range: 75 - 250 mm
source_file
Data_Meet_RMC               256
Combined_Msand_Dataset_1     35
plant_a_pdfs           8
Name: count, dtype: int64

rows above the IS 1199 (Part 2) 210 mm limit: 13  (kept - see open issue)
rows with 45-min slump: 256 (one plant, 5 designs - descriptive only)


## 6. Strength dataset — one row per mix, same mixes as workability

Each mix appears **once**. The target is the **28-day** mean of the individual cubes —
the acceptance age (IS 456 cl 15, Table 11). 7-day and other-age results are kept as
**reference columns only**. They are never training rows, and never features: a 7-day
result is measured after casting, so it does not exist when a mix is being designed.

Earlier versions gave each age its own row, which counted one mix two or three times
and inflated the apparent sample size.

In [7]:
MIX_ID = 'record_id'
assert MIX_ID in d.columns, f"'{MIX_ID}' column not found - set MIX_ID to the mix identifier"
assert d[MIX_ID].is_unique, "a mix id appears twice in the raw data - resolve before continuing"

def cube_stats(r, cols):
    """mean of the cubes present, how many, and the largest deviation of any cube from that mean (%)."""
    v = [pd.to_numeric(r.get(c), errors='coerce') for c in cols]
    v = [x for x in v if pd.notna(x)]
    if not v:
        return np.nan, 0, np.nan
    m = float(np.mean(v))
    dev = max(abs(x - m) for x in v) / m * 100 if len(v) > 1 else np.nan
    return m, len(v), dev

keep_cols = [c for c in work.columns if not c.startswith('cube_')]
recs = []
for _, r in work.iterrows():                       # ONLY the workability mixes
    fc28, n28, dev28 = cube_stats(r, ['cube_28d_1','cube_28d_2','cube_28d_3'])
    fc7,  n7,  _     = cube_stats(r, ['cube_7d_1','cube_7d_2','cube_7d_3'])
    oa = pd.to_numeric(r.get('cube_other_age'), errors='coerce')
    fco, no, _ = cube_stats(r, ['cube_other_1','cube_other_2','cube_other_3'])
    recs.append({**{c: r[c] for c in keep_cols},
                 'age_days': 28.0,                 # every row is the 28-day result
                 'fc_mpa': fc28, 'n_cubes': n28, 'cube_max_dev_pct': dev28,
                 'fc_7d_ref_mpa': fc7,             # reference only - NOT a feature
                 'other_age_days': oa, 'fc_other_ref_mpa': fco})   # reference only

strength = pd.DataFrame(recs)
no28 = strength.fc_mpa.isna()
if no28.any():
    print(f"workability mixes with no 28-day cubes (dropped from strength): {no28.sum()}")
    print("   ", strength.loc[no28, MIX_ID].tolist()[:20])
strength = strength[~no28].reset_index(drop=True)

left_out = sorted(set(d[MIX_ID]) - set(work[MIX_ID]))
print(f"\nstrength: {len(strength)} mixes (one row each)")
print(f"  mixes with cubes but no cone slump, left out by project choice: {len(left_out)}")
print(f"  with a 7-day reference value: {strength.fc_7d_ref_mpa.notna().sum()}")
print(f"  with an other-age reference:  {strength.fc_other_ref_mpa.notna().sum()}")


strength: 299 mixes (one row each)
  mixes with cubes but no cone slump, left out by project choice: 45
  with a 7-day reference value: 284
  with an other-age reference:  8


### Cube consistency check — each cube within 15% of the sample mean

Rule as read earlier in this project: IS 456 cl 15.4 and IS 516 (Part 1/Sec 1):2021 cl 3.6 —
if an individual cube varies by more than ±15% from the average of the sample, the sample is
invalid, unless the two-closest-values provision of IS 516:2021 cl 3.6 is applied.

**Corrected.** The earlier version measured (highest − lowest) / mean, which is not the rule and
over-flags (cubes of 90, 100, 110 give 20% "spread" although every cube is within 10%).
Rows are **flagged, not removed** — whether to drop them or apply the two-closest provision is an
engineering decision, recorded per row.

In [8]:
strength['cube_15pct_ok'] = np.where(strength.cube_max_dev_pct.isna(), np.nan,
                                     strength.cube_max_dev_pct <= 15)
bad = strength.cube_max_dev_pct > 15
print(f"mixes with a 28-day cube beyond 15% of the mean: {bad.sum()} / {strength.cube_max_dev_pct.notna().sum()}  (flagged, kept)")
print(f"mixes with fewer than 2 cubes (check not possible): {strength.cube_max_dev_pct.isna().sum()}")
print(f"mean largest deviation: {strength.cube_max_dev_pct.mean():.2f}%")

mixes with a 28-day cube beyond 15% of the mean: 0 / 299  (flagged, kept)
mixes with fewer than 2 cubes (check not possible): 0
mean largest deviation: 2.28%


## 7. Save


In [9]:
STRENGTH_FEATURES = ['cement_kg_m3','flyash_kg_m3','ggbs_kg_m3','water_kg_m3','wb_ratio',
                     'coarse_agg_kg_m3','fine_agg_kg_m3','sp_kg_m3','sp_pct_binder',
                     'scm_frac','agg_binder_ratio','fine_frac_of_agg']        # log_age removed: all rows are 28 days
WORK_FEATURES     = ['cement_kg_m3','flyash_kg_m3','ggbs_kg_m3','water_kg_m3','wb_ratio',
                     'coarse_agg_kg_m3','fine_agg_kg_m3','sp_kg_m3','sp_pct_binder',
                     'scm_frac','agg_binder_ratio','fine_frac_of_agg','paste_frac',
                     'sand_absorption']

# ---- the two files must describe the same mixes, once each ----
assert strength[MIX_ID].is_unique and work[MIX_ID].is_unique
assert set(strength[MIX_ID]) <= set(work[MIX_ID]), "strength contains a mix that workability does not"

strength.to_csv('03_strength_dataset.csv', index=False)
work.to_csv('04_workability_dataset.csv', index=False)

print(f"03_strength_dataset.csv     {len(strength)} rows (one per mix), target fc_mpa = 28-day")
print(f"04_workability_dataset.csv  {len(work)} rows (one per mix), target slump_mm")
print(f"same mixes in both: {len(set(strength[MIX_ID]) & set(work[MIX_ID]))}")
print(f"designs: strength {strength.design_grp.nunique()}, workability {work.design_grp.nunique()}")
print(f"\nstrength complete on features: {strength.dropna(subset=STRENGTH_FEATURES+['fc_mpa']).shape[0]}")
print(f"workability complete on features: {work.dropna(subset=WORK_FEATURES+['slump_mm']).shape[0]}")

03_strength_dataset.csv     299 rows (one per mix), target fc_mpa = 28-day
04_workability_dataset.csv  299 rows (one per mix), target slump_mm
same mixes in both: 299
designs: strength 41, workability 41

strength complete on features: 299
workability complete on features: 299
